In [ ]:
from option_finder import *
from option_data_plotter import *
try:
    logger
except NameError:
    logger = get_rotating_logger("jupyter", f'logs/call_options.log')

chain_dir = 'chain'
quotes_dir = 'quotes'
data_dir = 'data'
cookie_file = 'cookie.txt'
self = OptionFinder(logger, chain_dir=chain_dir, report_dir=data_dir)

In [ ]:
age_dict = dict([(os.path.basename(f), time.time() - os.path.getmtime(f)) for f in glob(os.path.join(self.chain_dir, '*'))])
symlist = sorted([k for k, v in age_dict.items() if v <= 60], key=age_dict.get)
if len(symlist) > 0:
    print(f'{len(symlist)} symbols, age: {age_dict[symlist[0]]:.01f}" {age_dict[symlist[-1]]:.01f}", {" ".join(symlist)}')
else:
    print('No update in the past 15 minutes.')

### Read downloaded data, check the data age chart to make sure the data are fresh

In [ ]:
df_quotes, df_shortint, df_vola = self.get_quote_df(symlist)
_t0 = time.time()
_df = self.build_option_df(symlist)
_t1 = time.time()
print(f'build_option_df {_t1 - _t0:.1f} seconds')
dfcp = self.concat_put_call_options(_df)
dfcp = bucketize_dte(add_moneyness_columns(dfcp))
_t2 = time.time()
print(f'concat_put_call_options {_t2 - _t1:.1f} seconds')
px.bar(check_data_age(_df), y=['load_age', 'quote_age'], barmode='group', title=f"Data Ages", width=60*len(symlist), height=300).show()
print(f'px.bar {time.time() - _t2:.1f} seconds')
dfcp.loc[:, ['dte', 'expDt']].groupby('dte').first().head(24).tail(20).T

In [ ]:
try:
    d2e = count_days_from_earning_reports(df_quotes)['earningDays'].to_dict()
except KeyError:
    d2e = {}
print('Days to E:', d2e)

In [ ]:
_g = dfcp[(dfcp.type=='C') & (dfcp.Bid >= 0.5) & (dfcp.OpenInterest >= 100)].groupby('symbol')
_df = pd.DataFrame({'mean spread': _g.pctSpread.mean(), 'median': _g.pctSpread.median()})
px.bar(_df.sort_values(by='mean spread'), barmode='group', width=60*len(symlist))

### Call Options: ignore no-bid or low open interest (minimum open interests is 100)

In [ ]:
dfc = compute_all_time_decay_metrics_for_symbols(dfcp, symlist, 'C', d2e, ignore_no_bid=True, exclude_0dte=True, oi_lb=100)
print('hdte_resid check:', dfc[(dfc.hdte_resid < dfc.resid) & (np.abs(dfc.hdte_resid - dfc.resid) >= 1e-6)].shape)

In [ ]:
hdte_resid_lb = 0.9
overpaid_ub = 0.05
spread_ub = 5
_filter = (dfc.dte >= 90) & (dfc.hdte_resid >= hdte_resid_lb) & (dfc.overpaid <= overpaid_ub) & (dfc.pctSpread <= spread_ub)
_filter = _filter & (dfc.symbol != 'TLT')
_dfc = dfc[_filter].drop(columns=['dth', 'dtz', 'dthr', 'dtzr']).sort_values(by='leverage', ascending=False)
_dfc.head(20)

In [ ]:
_filter = (dfc.pctSpread <= 10) & (dfc.strike <= dfc.lastPrice) & (dfc.dte >= 60)
px.scatter(dfc[_filter].sort_values(by='leverage', ascending=False).head(1000), x='hdte_resid', y='leverage', color='symbol', height=600)

In [ ]:
dfc[(dfc.symbol=='TSM') & (dfc.dte >= 90) & (dfc.strike <= dfc.lastPrice) & (dfc.leverage <= 8)].sort_values(by='leverage', ascending=False).head(20)

### The End